# ResNet-50 / ImageNet — technische check en pilot van twee epochs

**Doel:** eerst bevestigen dat het officiële Any-Precision-model met batch 256 op deze GPU uitvoerbaar is, daarna de looptijd van twee volledige ImageNet-epochs meten. De oorspronkelijke GitHub-configuratie gebruikt 80 epochs, SGD en startlearningrate 0,5. De definitieve 80-epoch-run begint later opnieuw vanaf nul.

Deze notebook wordt **onuitgevoerd** meegeleverd. Open bij voorkeur de kopie in de resultatenmap van `check`, `data` of `run` en kies **Run All**. Er wordt geen training gestart. Grafieken worden standaard niet opgeslagen.

In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# De runner vult dit in voor de kopie in de resultatenmap.
RUN_DIRECTORY = "C:\\Users\\renzo\\master-thesis-mx\\mx_runs\\anyprecision_imagenet_resnet50_pilot2\\20261030_145941_328517_check"
DATA_ROOT = Path(RUN_DIRECTORY).expanduser().resolve() if RUN_DIRECTORY else Path.cwd().resolve()
SAVE_FIGURES = False

sys.path.insert(0, str(DATA_ROOT))
import notebook_support as ns

plt.rcParams.update({"font.size": 11, "figure.facecolor": "white", "axes.facecolor": "white"})
run = ns.load_run(DATA_ROOT)
export_directory = None

def maybe_save(fig, name):
    global export_directory
    if fig is not None and SAVE_FIGURES:
        if export_directory is None:
            export_directory = DATA_ROOT / ("analysis_exports_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
            export_directory.mkdir(exist_ok=False)
        fig.savefig(export_directory / (name + ".png"), dpi=180, bbox_inches="tight")

print("Resultatenmap:", DATA_ROOT)
display(ns.run_summary(run))
for field in ("error", "message", "note"):
    if run["report"].get(field):
        print(field + ":", run["report"][field])

## 1. Wat wordt gelijk gehouden?

- Het meegeleverde `resnet50q`-model, vijf modi **W32/A32 → W8/A8 → W4/A4 → W2/A2 → W1/A1**, recursieve distillatie en één SGD-update per batch van 256 beelden.
- SGD met learningrate **0,5**, momentum **0,9**, weight decay **0,00001**; de learningrate daalt na epochs **45, 60 en 70**. Dit 80-epochschema wordt voor de pilot niet ingekort.
- FP32-mastergewichten en FP32-rekenkernels; quantisatie wordt in de forwards nagebootst. AMP/TF32 staan niet aan.
- Eén GPU verwerkt de vier oorspronkelijke deelbatches achter elkaar. Iedere BatchNorm-berekening ziet normaal 64 beelden; alleen de bufferupdates van de eerste deelbatch blijven per precisie bewaard, zoals bij `DataParallel` op vier GPU’s. De laatste, onvolledige batch wordt verdeeld zoals `torch.chunk`.

Dit is een gecontroleerde nabootsing van de oorspronkelijke batchverdeling. Een andere sommatievolgorde en moderne software/hardware betekenen dat bitexacte gelijkheid met de auteursrun niet wordt beloofd. De notebook rapporteert een technische PASS afzonderlijk van de gemeten accuracy.

## 2. Technische GPU-check

De input voor `check` is **synthetisch: 256 × 3 × 224 × 224**. De meting omvat alle vijf precisies, forward/backward en één SGD-update, na opwarming. Bij `serial_dp4` worden per precisie vier deelbatches verwerkt. De timings omvatten geen ImageNet-bestanden lezen, augmentatie of validatie.

Vanaf v1.1 wordt standaard alleen **`serial_dp4`** uitgevoerd: de gekozen trainingsroute met vier deelbatches van 64. `native256` krijgt dan **`SKIPPED`**. Die extra diagnose met alle 256 beelden tegelijk en BN over 256 kan expliciet worden ingeschakeld en start pas na een geslaagde seriële proef.

Elke GPU-proef krijgt standaard **300 seconden** voor opstart, opwarming en metingen samen. **`TIMEOUT` betekent alleen dat de proef binnen deze grens niet voltooid is; het bewijst geen OOM.** De tabel toont waar beschikbaar de laatste fase en verstreken tijd. Een tijdmelding van het hoofdproces is op zichzelf geen bewijs van nieuwe GPU-voortgang. Een OOM of TIMEOUT van de gekozen seriële route blokkeert de pilot.

Een overgeslagen, afgebroken of mislukte proef wordt niet als nul seconden of nul geheugen voorgesteld. Alleen een volledig geslaagde proef krijgt meetstaven. Extra synchronisatie en detailmeldingen zitten uitsluitend in de opwarming, buiten de twee gemeten stappen.


In [ ]:
memory = ns.memory_table(run)
if memory.empty:
    print("Geen geheugenmetingen in dit rapport. Gebruik hiervoor de notebook uit de check-resultatenmap.")
else:
    display(memory.drop(columns=["mode"]).round(3))
fig = ns.plot_memory(run)
maybe_save(fig, "gpu_geheugen_en_stapduur")
plt.show()

## 3. Dataset en uitvoering

De echte pilot gebruikt de volledige **ILSVRC2012-trainset (1.281.167 beelden)** en **validatieset (50.000 beelden)**, met 1.000 identieke WNID-klasmappen per split. De ongelabelde testsplit wordt niet gebruikt. Het pakket downloadt, verplaatst of herlabelt geen data.

Training: willekeurige crop naar 224 × 224, horizontaal spiegelen, RGB-tensor en ImageNet-normalisatie. Validatie: kortste zijde 256, centrale crop 224 × 224 en dezelfde normalisatie. Evaluatiebatch 64 verandert de bevroren BN-statistieken niet.

De data-audit controleert structuur, aantallen, bestandsnamen/groottes en een kleine steekproef decodes. Dat is geen volledige inhoudscontrole of onafhankelijke bevestiging van de validatielabels.

In [ ]:
manifest = None
for filename in ("dataset_manifest.json", "data_manifest.json"):
    candidate = DATA_ROOT / filename
    if candidate.is_file():
        manifest = json.loads(candidate.read_text(encoding="utf-8"))
        break
if manifest is None:
    print("Geen ImageNet-inventaris in deze map; dit is normaal voor de synthetische check.")
else:
    rows = []
    for split, info in manifest.get("splits", {}).items():
        rows.append({"Split": split, "Beelden": info.get("count"),
                     "Klassen": manifest.get("class_count"),
                     "Bestanden [GiB]": info.get("file_bytes", 0) / 2**30,
                     "Steekproefdecodes": len(info.get("decode_samples", [])),
                     "Pad/grootte SHA-256": info.get("filelist_sha256")})
    display(pd.DataFrame(rows))
    print(manifest.get("note", ""))

## 4. Volledig accuracyverloop van de pilot

Hier staan uitsluitend werkelijk gemeten ImageNet-validaties. Een synthetische check levert geen onderzoeksaccuracy. De pilot stopt na epoch 2; er worden geen resultaten voor epoch 80 geëxtrapoleerd.

In [ ]:
fig = ns.plot_accuracy_curves(run, "top1")
maybe_save(fig, "validatie_top1_verloop")
plt.show()

In [ ]:
fig = ns.plot_accuracy_curves(run, "top5")
maybe_save(fig, "validatie_top5_verloop")
plt.show()

## 5. Laatste checkpoint en checkpoint met de beste hoogste precisie

De selectie gebruikt de hoogste **W32/A32-validatie-top-1**. Bij een gelijke score blijft het vroegste checkpoint gekozen, zoals de strikte `>`-vergelijking in de oorspronkelijke trainer. Alle lagere precisies in die vergelijking komen uit **datzelfde checkpoint**; afzonderlijke maxima worden niet gecombineerd.

Als de laatste epoch ook de beste W32/A32-epoch is, verwijzen beide panelen naar hetzelfde checkpoint. Een lage pilotaccuracy is op zichzelf geen technische fout.

In [ ]:
checkpoints = ns.checkpoint_table(run)
if not checkpoints.empty:
    display(checkpoints.round(3))
fig = ns.plot_checkpoints(run)
maybe_save(fig, "laatste_en_beste_w32_checkpoint")
plt.show()
print("Epoch 80: niet uitgevoerd in dit pilotpakket.")

## 6. Looptijd en projectie naar 80 epochs

Deze projectie gebruikt echte epochduren, inclusief validatie en de geregistreerde checkpointtijd. Bij twee afgeronde epochs gebruiken we de eerste epoch plus 79 keer de tweede. Met slechts één epoch is de schatting voorlopig. Thermisch gedrag, stroomprofiel en datasetopslag kunnen de uiteindelijke duur beïnvloeden.

Eenmalige voorbereiding buiten de epochs is niet inbegrepen. De gemeten synthetische stapduur wordt niet voorgesteld als een gemeten volledige ImageNet-trainingsduur.

In [ ]:
times = ns.epoch_timing_table(run)
if times.empty:
    print("Er zijn nog geen echte epochduren beschikbaar.")
else:
    display(times.round(3))
    display(ns.runtime_projection(run).round(2))
fig = ns.plot_epoch_times(run)
maybe_save(fig, "epochduur_onderdelen")
plt.show()

## 7. GitHub-referentie voor de latere volledige training

Dit zijn de gepubliceerde resultaten voor **één gezamenlijk getraind ResNet-50-Any-model met recursieve distillatie**, bij het GitHub-recept van 80 epochs. Ze zijn een referentie voor de latere volledige training, geen verwachting voor epoch 2. De README vermeldt niet expliciet op welke epoch het gepubliceerde checkpoint is geselecteerd; we noemen deze cijfers daarom niet zonder bewijs “epoch 80”.

De GitHub-code verschilt van het paperrecept, onder meer in de startlearningrate (0,5 versus 0,3) en de FP32-activatie-/gewichtsbewerking.

In [ ]:
display(ns.reference_table())
print("Bron:", ns.REFERENCE_URL)

## 8. Wat besluit deze proef?

Een geslaagde technische check bevestigt dat de gekozen uitvoering een volledige trainingsstap kan uitvoeren binnen het beschikbare GPU-geheugen. De pilot controleert daarna de echte datapijplijn, training, validatie, checkpointing en looptijd. Twee epochs bewijzen nog geen reproductie van de gepubliceerde eindaccuracy.

Bewaar de pilotcheckpoints en deel `feedback_bundle.zip` uit de resultatenmap. De volledige 80-epoch-training wordt later met een nieuwe run vanaf de oorspronkelijke initialisatie gestart; dit pakket hervat de pilot niet tot epoch 80.

**Bronnen**

- [Officiële Any-Precision-code en resultaten](https://github.com/SHI-Labs/Any-Precision-DNNs)
- [ImageNet: officiële datasetomvang](https://ftp.image-net.org/download)
- [ImageNet CLS-LOC: dezelfde dataset als ILSVRC2012](https://image-net.org/challenges/LSVRC/2013/2013-downloads.php)
- [PyTorch 2.10: DataParallel en behoud van buffers op GPU 0](https://docs.pytorch.org/docs/2.10/generated/torch.nn.DataParallel.html)
- [PyTorch 2.10: BatchNorm2d](https://docs.pytorch.org/docs/2.10/generated/torch.nn.BatchNorm2d.html)